In [1]:
from tinydas.dataloader import DataLoader
from tinydas.dataset import DataSet
from tinygrad import Device 
from tinygrad.helpers import getenv

from typing import Optional

from tinygrad import TinyJit, dtypes, nn
from tinygrad.nn import Tensor

import random as rnd

from tqdm import trange

In [2]:
Tensor.manual_seed(574)
rnd.seed(574)

In [3]:
GPUS = [f'{Device.DEFAULT}:{i}' for i in range(getenv("GPUS", 2))]
GPUS

['CUDA:0', 'CUDA:1']

In [4]:
ds = DataSet(n = 10)

In [5]:
ds

(10, 625, 2137)

In [6]:
dl = DataLoader(ds, batch_size=2)

In [7]:
dl.batch_size, dl.current_index, dl.num_samples, dl.times, ds.shape

(2,
 0,
 10,
 <Tensor <LB CUDA (10, 625) double (<LoadOps.COPY: 3>, None)> on CUDA with grad None>,
 (10, 625, 2137))

In [8]:
def mse(X: Tensor, Y: Tensor):
    return Y.sub(X).square().mean()


class LL:
    def __init__(self, i: int, o: int, do: Optional[float] = None) -> None:
        self.net = [
            nn.Linear(i, o),
            Tensor.relu,
        ]
        if do:
            self.net.append(Tensor.dropout)

    def __call__(self, x: Tensor) -> Tensor:
        return x.sequential(self.net)


class AE:
    def __init__(self):
        self.M = 625
        self.N = 2137
        self.inp = self.M * self.N
        self.hidden = 2137
        self.latent = 128

        self.net = [
            LL(self.inp, self.hidden),
            LL(self.hidden, self.latent),
            
            LL(self.latent, self.hidden),
            nn.Linear(self.hidden, self.inp),
            Tensor.sigmoid,
        ]

    def __call__(self, x: Tensor) -> Tensor:
        return x.sequential(self.net)

    def predict(self, x: Tensor) -> Tensor:
        return self(x)

In [9]:
model = AE()

for k, x in nn.state.get_state_dict(model).items(): x.to_(GPUS)

In [10]:
optim = nn.optim.Adam(nn.state.get_parameters(model))

In [11]:
@TinyJit
def step():
    with Tensor.train():
        rl = 0.0
        for batch, _ in dl:
            data = batch.shard_(GPUS, axis=0).reshape(-1, model.inp)
            optim.zero_grad()
            l = model(data).sub(data).square().mean().backward()
            optim.step()
            rl += l
        return rl

In [ ]:
for epoch in trange(t := 5):
    l = step()
    print(f"Loss: {l:.3f}")
    
    

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
l = mse(data, out).item()


pred = model.predict(data)

print(f"Prediction: \n{pred.numpy().mean():.3f}")